# Hybrid Quantum-Classical ML for Early Breast Cancer Detection
### SIH 2026 (SIH26139): Hybrid Quantum Machine Learning Platform for Early Disease Detection

This demo notebook demonstrates the end-to-end workflow:
1. Loading and preprocessing gene expression data
2. Patient-level train/validation split
3. Differential expression (t-test) and Mutual Information ranking
4. Constructing and training the Hybrid Quantum-Classical model (`AngleEmbedding` + `StronglyEntanglingLayers`)
5. Comparing against classical baselines and parameter-matched ablation
6. Inspecting explainability with SHAP and permutation importance

In [ ]:
import yaml
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data import generate_synthetic_data
from src.features import patient_level_split, remove_low_quality_genes, select_features_fold, scale_features
from src.models_quantum import HybridQuantumModel, ClassicalAblationModel, train_torch_model, compute_class_weights
from src.models_classical import train_svm, train_random_forest, train_xgboost, get_classical_predictions
from src.evaluate import compute_metrics, find_optimal_threshold

## 1. Load Synthetic TCGA-BRCA Data

In [ ]:
expr_df, labels, patient_ids = generate_synthetic_data(n_tumor=120, n_normal=30, n_genes=500, seed=42)
print(f"Expression matrix shape: {expr_df.shape}")
print(f"Labels count:\n{labels.value_counts()}")

## 2. Preprocessing & Patient-Level Splitting

In [ ]:
# Filter low variance
expr_filtered = remove_low_quality_genes(expr_df, variance_threshold=0.1, min_expression=1.0)

# Patient-level stratified split (80% train+val, 20% held-out test)
X_train, X_test, y_train, y_test, pid_train, pid_test = patient_level_split(
    expr_filtered, labels, patient_ids, test_size=0.2, seed=42
)
print(f"Train samples: {len(X_train)}, Test samples: {len(X_test)}")
assert set(pid_train).isdisjoint(set(pid_test)), "Leakage detected!"

## 3. In-Fold Feature Selection

In [ ]:
# Top 8 genes selected strictly on training portion
selected_genes = select_features_fold(
    X_train, y_train, n_de_genes=200, n_final_genes=8, seed=42
)
print(f"Selected 8 Qubit Genes:\n{selected_genes}")

## 4. Scaling Features to [0, pi]

In [ ]:
X_tr_raw = X_train[selected_genes].values
X_te_raw = X_test[selected_genes].values

X_tr_s, _, X_te_s, scaler = scale_features(X_tr_raw, X_tr_raw, X_te_raw)
print(f"Feature range train: [{X_tr_s.min():.2f}, {X_tr_s.max():.2f}]")

## 5. Train Hybrid Quantum Model

In [ ]:
hybrid_model = HybridQuantumModel(n_qubits=8, n_layers=2)
print("Parameter Counts:", hybrid_model.count_parameters())

pos_weight = compute_class_weights(y_train.values)
trained_model, history = train_torch_model(
    hybrid_model, X_tr_s, y_train.values, X_te_s, y_test.values,
    lr=0.02, batch_size=16, max_epochs=10, patience=5, pos_weight=pos_weight, seed=42
)

## 6. Inspect Results

In [ ]:
with torch.no_grad():
    preds = trained_model(torch.tensor(X_te_s, dtype=torch.float32)).numpy().flatten()
metrics = compute_metrics(y_test.values, preds, threshold=0.5)
print("Test Set Metrics:", metrics)